In [3]:
#드롭아웃별 for문
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import csv

# ====== GPU/CPU 설정 ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")

# ====== 데이터 경로 설정 ======
dataset_path = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/data/camter_tent_data"

# ====== 공통 데이터 증강 및 전처리 ======
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ====== 학습 및 검증 파라미터 ======
BATCH_SIZE = 32
EPOCHS = 300

# ====== 드롭아웃 값 리스트 ======
dropout_rates = [0.2, 0.3, 0.5]

# ====== 데이터셋 로드 및 분할 ======
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ====== 드롭아웃 값 별 루프 ======
for dropout_rate in dropout_rates:
    print(f"\n===== 드롭아웃: {dropout_rate} 학습 시작 =====")

    # EfficientNetB0 모델 로드 및 수정
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    num_classes = len(dataset.classes)
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate),  # 드롭아웃 적용
        nn.Linear(model.classifier[1].in_features, num_classes)
    )
    model = model.to(device)

    # 손실 함수 및 옵티마이저 설정
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # 학습 및 검증 저장 리스트 초기화
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    # ====== 학습 및 검증 ======
    for epoch in range(EPOCHS):
        # ===== 학습 =====
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

        train_accuracy = correct_train / total_train
        train_losses.append(running_loss / len(train_loader))
        train_accuracies.append(train_accuracy)

        # ===== 검증 =====
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)

                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)

        val_accuracy = correct_val / total_val
        val_losses.append(val_loss / len(val_loader))
        val_accuracies.append(val_accuracy)

        print((
            f"에폭 [{epoch + 1}/{EPOCHS}] - "
            f"학습 손실값: {running_loss/len(train_loader):.4f} - "
            f"학습 정확도: {train_accuracy:.4f} - "
            f"검증 손실값: {val_loss/len(val_loader):.4f} - "
            f"검증 정확도: {val_accuracy:.4f}"
        ))

    # ====== 결과 CSV 저장 ======
    csv_dir = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/csv"
    os.makedirs(csv_dir, exist_ok=True)

    csv_path = os.path.join(csv_dir, f"training_results_dropout_{dropout_rate}.csv")
    with open(csv_path, mode='w', newline='', encoding='utf-8-sig') as file:
        writer = csv.writer(file)
        writer.writerow(["Epoch", "Train Loss", "Valid Loss", "Train Accuracy", "Valid Accuracy"])
        for epoch in range(EPOCHS):
            writer.writerow([
                epoch + 1,
                train_losses[epoch],
                val_losses[epoch],
                train_accuracies[epoch],
                val_accuracies[epoch]
            ])

    print(f"드롭아웃 {dropout_rate} 결과가 '{csv_path}'에 저장되었습니다.")


사용 중인 디바이스: cuda

===== 드롭아웃: 0.2 학습 시작 =====
에폭 [1/300] - 학습 손실값: 2.1249 - 학습 정확도: 0.2762 - 검증 손실값: 1.8612 - 검증 정확도: 0.3636
에폭 [2/300] - 학습 손실값: 1.5078 - 학습 정확도: 0.4856 - 검증 손실값: 1.6149 - 검증 정확도: 0.4424
에폭 [3/300] - 학습 손실값: 1.1813 - 학습 정확도: 0.5903 - 검증 손실값: 1.7087 - 검증 정확도: 0.5152
에폭 [4/300] - 학습 손실값: 1.0539 - 학습 정확도: 0.6206 - 검증 손실값: 1.2469 - 검증 정확도: 0.5758
에폭 [5/300] - 학습 손실값: 0.9041 - 학습 정확도: 0.6662 - 검증 손실값: 1.4183 - 검증 정확도: 0.5152
에폭 [6/300] - 학습 손실값: 0.8403 - 학습 정확도: 0.6935 - 검증 손실값: 1.3338 - 검증 정확도: 0.5576
에폭 [7/300] - 학습 손실값: 0.8344 - 학습 정확도: 0.6631 - 검증 손실값: 1.5379 - 검증 정확도: 0.4909
에폭 [8/300] - 학습 손실값: 0.8380 - 학습 정확도: 0.6995 - 검증 손실값: 1.3338 - 검증 정확도: 0.6000
에폭 [9/300] - 학습 손실값: 0.7869 - 학습 정확도: 0.6904 - 검증 손실값: 1.3607 - 검증 정확도: 0.5636
에폭 [10/300] - 학습 손실값: 0.7514 - 학습 정확도: 0.7132 - 검증 손실값: 1.3916 - 검증 정확도: 0.5455
에폭 [11/300] - 학습 손실값: 0.7072 - 학습 정확도: 0.7405 - 검증 손실값: 1.4686 - 검증 정확도: 0.5455
에폭 [12/300] - 학습 손실값: 0.7244 - 학습 정확도: 0.7117 - 검증 손실값: 1.1060 - 검증 정확도: 0.6242
에폭 

In [38]:
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import recall_score
import pandas as pd

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 정의
model = models.efficientnet_b0(weights=None)  # 사전 학습 가중치 사용 안 함
num_classes = 6 # 데이터셋의 클래스 수 (학습 당시 사용한 값)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, num_classes)  # 저장된 모델 구조 반영
)

# 저장된 가중치 로드
model_path = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/models/efficientnet_tent_model2.pth"
try:
    model.load_state_dict(torch.load(model_path))  # 저장된 가중치 로드
    print("가중치를 성공적으로 로드했습니다.")
except RuntimeError as e:
    print(f"가중치 로드 중 오류 발생: {e}")
    raise

# 평가 모드 전환
model = model.to(device)
model.eval()

# 리콜 계산 함수
def calculate_recall(loader, dataset_type="validation"):
    all_labels, all_predictions = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    # 리콜 계산
    recall = recall_score(all_labels, all_predictions, average=None)
    recall_df = pd.DataFrame({
        'Class': range(len(recall)),
        'Recall': recall
    })

    # CSV로 저장
    output_path = f"C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/csv/{dataset_type}_recall.csv"
    recall_df.to_csv(output_path, index=False)

    print(f"{dataset_type.capitalize()} 데이터 재현율:")
    print(recall_df)

# 훈련 데이터 리콜 계산
calculate_recall(train_loader, "train")

# 검증 데이터 리콜 계산
calculate_recall(val_loader, "validation")


C:\Users\user\AppData\Local\Temp\ipykernel_9688\3542172256.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))  # 저장된 가중치 로드


가중치를 성공적으로 로드했습니다.
Train 데이터 재현율:
   Class    Recall
0      0  0.857143
1      1  0.803738
2      2  0.941176
3      3  0.894737
4      4  0.886364
5      5  0.934426
Validation 데이터 재현율:
   Class    Recall
0      0  0.818182
1      1  0.851852
2      2  0.933333
3      3  0.928571
4      4  0.900000
5      5  0.950000
